# Evolutionary Tournament — Data Analysis

Works **without Stockfish**: the reference signal is the built-in classical
static evaluator (`EvalAgent`).  To use Stockfish, set `STOCKFISH_EXECUTABLE`
to its path *before* starting Jupyter and the rest adapts automatically.

Run Jupyter from the `cubist` repo root so `import evolutionary_tournament` resolves.

In [ ]:
%matplotlib inline
from __future__ import annotations
import math, os, random, sys
from pathlib import Path

import chess
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path(os.getcwd())
if not (REPO / 'evolutionary_tournament' / 'evolution.py').exists():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)
print('Repo root:', REPO.resolve())

from evolutionary_tournament.__main__ import _DEFAULT_FENS
from evolutionary_tournament.evolution import run_evolution
from evolutionary_tournament.tunable_classical import TunableWeights, evaluate_tunable
from evolutionary_tournament.arena import play
from evolutionary_tournament.engines import ClassicalEngine, Berserker2Engine
from evolutionary_tournament.ground_truth import _try_open_stockfish

stockfish_available = _try_open_stockfish() is not None
print('Stockfish available:', stockfish_available)
print('Reference signal:',
      'Stockfish' if stockfish_available
      else 'built-in classical EvalAgent (no Stockfish)')


## 1 · Evolution

Optimises scalar weights `w_mat` (material) and `w_pst` (piece-square tables)
by maximising Pearson correlation with the reference evaluator across a fixed
FEN set.  Without Stockfish the reference is the built-in `EvalAgent`, so
weights converge quickly to the default (rho ≈ 1 from round 1 — this is expected).
The experiment becomes richer when Stockfish provides a stronger reference signal.

In [ ]:
N_ROUNDS = 4
POP      = 8
RNG_SEED = 42

print(f'Evolving: {POP} individuals x {N_ROUNDS} rounds ...')
stats = run_evolution(_DEFAULT_FENS, population=POP, rounds=N_ROUNDS,
                      rng=random.Random(RNG_SEED))
print('Done.')

evo_rows = []
for s in stats:
    w = s.best_weights
    evo_rows.append({
        'round':    s.round,
        'best_rho': round(s.best_fitness, 5),
        'mean_rho': round(s.mean_fitness, 5),
        'w_mat':    round(w.w_mat, 4),
        'w_pst':    round(w.w_pst, 4),
        'w_bias':   round(w.w_bias, 3),
    })

evo_df = pd.DataFrame(evo_rows).set_index('round')
best_weights    = stats[-1].best_weights
default_weights = TunableWeights(1.0, 1.0, 0.0)
print(f'Best weights: w_mat={best_weights.w_mat:.4f}  '
      f'w_pst={best_weights.w_pst:.4f}  bias={best_weights.w_bias:.3f}')
evo_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

ax = axes[0]
ax.plot(evo_df.index, evo_df['best_rho'], marker='o', label='best rho')
ax.plot(evo_df.index, evo_df['mean_rho'], marker='s', linestyle='--', label='mean rho')
ax.set_xlabel('Round'); ax.set_ylabel('Pearson rho')
ax.set_title('Fitness over evolution'); ax.legend(); ax.set_ylim(0, 1.05)

ax = axes[1]
ax.plot(evo_df.index, evo_df['w_mat'], marker='o', label='w_mat')
ax.plot(evo_df.index, evo_df['w_pst'], marker='s', linestyle='--', label='w_pst')
ax.axhline(1.0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('Round'); ax.set_ylabel('Weight value')
ax.set_title('Evolved weights'); ax.legend()

ax = axes[2]
ax.plot(evo_df.index, evo_df['w_bias'], marker='^', color='tab:green')
ax.axhline(0.0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('Round'); ax.set_ylabel('Bias (cp)')
ax.set_title('Evolved bias')

plt.tight_layout()
plt.show()


In [ ]:
# Compare raw centipawn scores: default vs evolved weights across test positions
cmp_rows = []
for fen in _DEFAULT_FENS:
    b = chess.Board(fen=fen)
    cmp_rows.append({
        'position':   fen[:28] + '...',
        'default_cp': evaluate_tunable(b, default_weights),
        'evolved_cp': evaluate_tunable(b, best_weights),
    })
cmp_df = pd.DataFrame(cmp_rows)
cmp_df['delta_cp'] = cmp_df['evolved_cp'] - cmp_df['default_cp']

fig, ax = plt.subplots(figsize=(9, 3.5))
idx = np.arange(len(cmp_df))
w_ = 0.35
ax.bar(idx - w_/2, cmp_df['default_cp'], w_, label='default (w=1,1)', color='#457b9d')
ax.bar(idx + w_/2, cmp_df['evolved_cp'], w_, label='evolved',         color='#e9c46a')
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xticks(idx, [f'pos {i}' for i in range(len(cmp_df))])
ax.set_ylabel('Centipawns (White view)')
ax.set_title('Static eval: default vs evolved weights')
ax.legend()
plt.tight_layout()
plt.show()

print('Mean |delta| across positions:', round(cmp_df['delta_cp'].abs().mean(), 1), 'cp')
cmp_df


## 2 · Head-to-head games

Three pairs; colors alternate each game so each side plays half as White.

| # | Player A | Player B |
|---|---|---|
| 1 | default classical | Berserker2 |
| 2 | evolved classical | default classical |
| 3 | evolved classical | Berserker2 |

> **`GAMES = 1`** for fast debugging.  Set to **20** for real statistics
> (≈8 min) or **100** for publication-level results (≈40 min).

In [ ]:
# ── tuneable ────────────────────────────────────────────────────────────
GAMES     = 1     # increase to 20+ for real stats
DEPTH_C   = 2
DEPTH_B2  = 2
MAX_PLIES = 40


def _outcome(board, raw, a_is_white):
    if raw and raw.endswith('*'):
        return 'draw'
    if not board.is_game_over():
        return 'draw'
    r = board.result(claim_draw=True)
    if r == '1/2-1/2':
        return 'draw'
    if a_is_white:
        return 'win' if r == '1-0' else 'loss'
    return 'win' if r == '0-1' else 'loss'


def run_pair(label, make_a, make_b, n=GAMES):
    w = d = l = 0
    for g in range(n):
        a_white = (g % 2 == 0)
        white = make_a() if a_white else make_b()
        black = make_b() if a_white else make_a()
        board, res = play(white, black, max_plies=MAX_PLIES)
        o = _outcome(board, res, a_is_white=a_white)
        if o == 'win':    w += 1
        elif o == 'loss': l += 1
        else:             d += 1
        print(f'  [{label}] game {g+1}/{n}: {o}')
    tot = w + d + l
    return {'pair': label, 'W': w, 'D': d, 'L': l,
            'games': tot, 'win_rate': w / tot if tot else 0.0}


mk_c  = lambda wt: ClassicalEngine(depth=DEPTH_C, weights=wt)
mk_b2 = lambda:    Berserker2Engine(depth=DEPTH_B2)

print(f'Running {GAMES} game(s) per pair (depth C={DEPTH_C}, B2={DEPTH_B2}) ...')
rows = [
    run_pair('default_classical vs berserker2',
             lambda: mk_c(default_weights), mk_b2),
    run_pair('evolved_classical vs default_classical',
             lambda: mk_c(best_weights), lambda: mk_c(default_weights)),
    run_pair('evolved_classical vs berserker2',
             lambda: mk_c(best_weights), mk_b2),
]
results_df = pd.DataFrame(rows).set_index('pair')
print()
results_df


In [ ]:
from IPython.display import Markdown, display

def wilson_95(wins, n):
    if n <= 0:
        return float('nan'), float('nan')
    p, z = wins / n, 1.96
    z2 = z * z
    denom  = 1 + z2 / n
    center = (p + z2 / (2 * n)) / denom
    half   = z * math.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / denom
    return center - half, center + half

ci_data = {}
for pair, row in results_df.iterrows():
    lo, hi = wilson_95(int(row['W']), int(row['games']))
    ci_data[pair] = (lo, hi)

lines = ['**Win-rate summary (Wilson 95% CI):**', '']
for pair, row in results_df.iterrows():
    lo, hi = ci_data[pair]
    lo_s = f'{lo*100:.1f}' if not math.isnan(lo) else 'n/a'
    hi_s = f'{hi*100:.1f}' if not math.isnan(hi) else 'n/a'
    lines.append(
        f'- **{pair}**: W/D/L = {int(row["W"])}/{int(row["D"])}/{int(row["L"])}  '
        f'win rate {row["win_rate"]*100:.1f}%  (95% CI {lo_s}%-{hi_s}%)'
    )
if GAMES < 10:
    lines.append('')
    lines.append('> *Sample too small for meaningful CIs — increase `GAMES`.*')
display(Markdown('\n'.join(lines)))
# add CI columns to results_df for charting
results_df['ci_lo'] = [ci_data[p][0] for p in results_df.index]
results_df['ci_hi'] = [ci_data[p][1] for p in results_df.index]
results_df[['W','D','L','win_rate','ci_lo','ci_hi']].round(3)


In [ ]:
short_labels = [p.replace(' vs ', '\nvs ') for p in results_df.index]
x = np.arange(len(short_labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# stacked bar W/D/L
ax = axes[0]
ax.bar(x, results_df['W'], label='A wins',   color='#2a9d8f')
ax.bar(x, results_df['D'], bottom=results_df['W'],
       label='Draws', color='#6c757d')
ax.bar(x, results_df['L'], bottom=results_df['W']+results_df['D'],
       label='A losses', color='#e76f51')
ax.set_xticks(x, short_labels, fontsize=8)
ax.set_ylabel('Games')
ax.set_title(f'W/D/L ({GAMES} game(s) per pair, depth C={DEPTH_C} B2={DEPTH_B2})')
ax.legend()

# win rate + CI
ax = axes[1]
wr = results_df['win_rate'].to_numpy()
err_lo = np.nan_to_num(wr - results_df['ci_lo'].to_numpy())
err_hi = np.nan_to_num(results_df['ci_hi'].to_numpy() - wr)
ax.bar(x, wr, color='#457b9d', yerr=[err_lo, err_hi], capsize=6)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
ax.set_xticks(x, short_labels, fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel('Win rate (A)')
ax.set_title('Win rate with Wilson 95% CI')

plt.tight_layout()
plt.show()


## Summary

| Parameter | Value |
|---|---|
| Reference evaluator | `EvalAgent` (built-in, no Stockfish) |
| Evolution: rounds × population | `N_ROUNDS` × `POP` |
| Games per pair | `GAMES` |
| Search depth | `DEPTH_C` / `DEPTH_B2` |
| Max plies per game | `MAX_PLIES` |

**Increase `GAMES`** for real statistics; **set `STOCKFISH_EXECUTABLE`** in the
environment before starting Jupyter to enable a stronger reference signal.